# Praktikum 6: Information Extraction from Recipts
In these exercises you will explore a real task: extracting information from scanned receipts!

In [ ]:
import pandas as pd
import ollama

## 1. Load the SROIE dataset

We start by loading the SROIE receipt dataset from a JSON file.
Each row corresponds to **one receipt**, with:

- `transcription` → OCR text (model input)
- `company`, `date`, `address`, `total` → gold fields (ground truth)

This structured format makes it easy to iterate over receipts and evaluate extraction performance.


````bash
mkdir -p data/sroie

curl -L \
  https://hsbi.sciebo.de/s/En8GaFrErkiyPNk/download/icdar-2019-sroie.json \
  -o data/sroie/icdar-2019-sroie.json
````

In [ ]:
import json

PATH = "../../data/sroie/icdar-2019-sroie.json"

with open(PATH, "r", encoding="utf-8") as f:
    sroie = json.load(f)

FIELDS = ["company", "date", "address", "total"] # fields to be extracted

items = sroie["items"]
items[0].keys()

In [ ]:
df = pd.json_normalize(sroie["items"])  # expands nested dicts into columns like fields.company
df.head(2)

In [ ]:
receipt = df.iloc[0]
print(receipt["transcription"])
print("\n=== gold data ===")
print ("Company: ", receipt["fields.company"] )
print ("Date: ", receipt["fields.date"] )
print ("Address: ", receipt["fields.address"] )

## 2. Generate with metrics
The helper below help us run the prompt through a given model while computing relevant performance metrics.

In [ ]:
import time

def generate_with_metrics(model, prompt, stream=False, print_live=True, **kwargs):
    t0 = time.time()

    text = ""
    first_token_time = None
    last = None  # we'll keep the last response/chunk (often contains stats)

    if stream:
        for chunk in ollama.generate(model=model, prompt=prompt, stream=True, **kwargs):
            if first_token_time is None:
                first_token_time = time.time()

            piece = chunk.get("response", "")
            if piece:
                text += piece
                if print_live:
                    print(piece, end="", flush=True)

            last = chunk

        if print_live:
            print()
    else:
        last = ollama.generate(model=model, prompt=prompt, stream=False, **kwargs)
        text = last.get("response", "")

    t1 = time.time()

    # Stats (sometimes present depending on server/version)
    prompt_tokens = last.get("prompt_eval_count")
    gen_tokens = last.get("eval_count")

    prompt_ns = last.get("prompt_eval_duration")  # nanoseconds
    gen_ns = last.get("eval_duration")            # nanoseconds

    prompt_s = (prompt_ns / 1e9) if prompt_ns else None
    gen_s = (gen_ns / 1e9) if gen_ns else None

    metrics = {
        "wall_s": t1 - t0,
        "ttft_s": (first_token_time - t0) if first_token_time else None,
        "prompt_tokens": prompt_tokens,
        "gen_tokens": gen_tokens,
        "prompt_tok_s": (prompt_tokens / prompt_s) if (prompt_tokens and prompt_s) else None,
        "gen_tok_s": (gen_tokens / gen_s) if (gen_tokens and gen_s) else None,
    }

    return text, metrics

## 3. Task-specific metrics
To evaluate receipt information extraction, we measure two complementary aspects:

### 3.1 Field-level extraction quality (EM + F1)

For each target field (`company`, `date`, `address`, `total`) we compute:

- **Exact Match (EM)**  
  The predicted field value matches the gold value exactly (after normalization).

- **Token-level F1**  
  Measures partial overlap between predicted and gold tokens.  
  This is more forgiving than EM and rewards partially correct extractions.

We also compute:

- **`all_fields_em`**  
  Equals 1 only if **all fields** are exact matches for a receipt (strict score).


In [ ]:
import re, string
from collections import Counter

def normalize(s: str) -> str:
    s = (s or "").lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = " ".join(s.split())
    return s

def f1_score(pred: str, gold: str) -> float:
    pred_toks = normalize(pred).split()
    gold_toks = normalize(gold).split()
    if not pred_toks and not gold_toks:
        return 1.0
    if not pred_toks or not gold_toks:
        return 0.0
    common = Counter(pred_toks) & Counter(gold_toks)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return 2 * precision * recall / (precision + recall)

def exact_match(pred: str, gold: str) -> float:
    return 1.0 if normalize(pred) == normalize(gold) else 0.0


In [ ]:

def receipt_metrics(pred_fields: dict, gold_fields: dict, fields=FIELDS):
    """
    Returns:
      - per-field EM and F1 (dict)
      - all_fields_em: 1 if ALL fields exact-match, else 0
    """
    out = {}
    all_ok = True

    for k in fields:
        pred = (pred_fields.get(k, "NONE") or "NONE").strip()
        gold = (gold_fields.get(k, "NONE") or "NONE").strip()

        em = exact_match(pred, gold)
        f1 = f1_score(pred, gold)

        out[f"em_{k}"] = em
        out[f"f1_{k}"] = f1
        if em < 1.0:
            all_ok = False

    out["all_fields_em"] = 1.0 if all_ok else 0.0
    return out

### 3.2 Behavior metrics (decision reliability)

In addition to span quality, we measure how well the model decides when to answer:

- **Hallucination rate**  
  The gold field is `NONE`, but the model outputs a value anyway.

- **False abstention rate**  
  The gold field has a value, but the model outputs `NONE`.

- **Attempt rate**  
  Fraction of receipts where the model outputs something other than `NONE`.

In [ ]:
def behavior_metrics_receipts(results_df, fields=FIELDS):
    """
    Decision-level behavior metrics per field + overall average.
    Assumes results_df has columns: pred_<field>, gold_<field>
    """
    def norm_none(x):
        x = (x or "").strip().upper()
        x = x.translate(str.maketrans("", "", string.punctuation))
        return x

    summary = {}
    for k in fields:
        pred = results_df[f"pred_{k}"].apply(norm_none)
        gold = results_df[f"gold_{k}"].apply(norm_none)

        gold_missing = gold.eq("NONE")
        gold_present = ~gold_missing

        halluc = gold_missing & ~pred.eq("NONE")
        false_abstain = gold_present & pred.eq("NONE")
        attempted = ~pred.eq("NONE")

        summary[f"hallucination_rate_{k}"] = halluc.sum() / gold_missing.sum() if gold_missing.sum() else 0.0
        summary[f"false_abstention_rate_{k}"] = false_abstain.sum() / gold_present.sum() if gold_present.sum() else 0.0
        summary[f"attempt_rate_{k}"] = attempted.sum() / len(results_df) if len(results_df) else 0.0

    # simple overall averages
    summary["hallucination_rate_avg"] = sum(summary[f"hallucination_rate_{k}"] for k in fields) / len(fields)
    summary["false_abstention_rate_avg"] = sum(summary[f"false_abstention_rate_{k}"] for k in fields) / len(fields)
    summary["attempt_rate_avg"] = sum(summary[f"attempt_rate_{k}"] for k in fields) / len(fields)

    return summary

## 4. Defining the prompt
We define the prompt and helper to parse the output.

In [ ]:
import json
import re

def make_receipt_prompt(transcription: str) -> str:
    return f"""Extract the following fields from the receipt text.
Use ONLY information that appears in the receipt.
Return JSON only (no extra text, no markdown).
If a field is missing, use the string "NONE".

Fields: {", ".join(FIELDS)}

Receipt text:
{transcription}

JSON:"""


def parse_json_like(text: str) -> dict:
    """
    Try to parse model output as JSON.
    Returns dict; if parsing fails, returns {}.
    """
    text = text.strip()
    # direct attempt
    try:
        return json.loads(text)
    except Exception:
        pass

    return {}


## 5. Running the task
This helper facilitates running the task over the dataset.

In [ ]:

def run_receipt_task(model, data, n=50, stream=False, options=None):
    results = []
    options = options or {"temperature": 0.0, "num_predict": 200}

    for i, row in enumerate(data[:n]):
        prompt = make_receipt_prompt(row["transcription"])

        pred_text, perf = generate_with_metrics(
            model=model,
            prompt=prompt,
            stream=stream,
            print_live=False,
            options=options
        )

        pred_obj = parse_json_like(pred_text)
        gold = row["fields"]

        # ensure we always have all expected keys
        pred_fields = {k: (pred_obj.get(k, "NONE") if isinstance(pred_obj, dict) else "NONE") for k in FIELDS}

        metrics = receipt_metrics(pred_fields, gold, fields=FIELDS)

        results.append({
            "id": row["id"],
            "source_key": row.get("source_key", ""),
            "transcription": row["transcription"],
            "pred_raw": pred_text,
            **{f"pred_{k}": pred_fields.get(k, "NONE") for k in FIELDS},
            **{f"gold_{k}": gold.get(k, "NONE") for k in FIELDS},
            **metrics,
            **perf
        })

        if (i + 1) % 10 == 0:
            print(f"{i+1}/{n} done")

    results = pd.DataFrame(results)

    # aggregate quality metrics
    summary = {
        "model": model,
        "n": len(results),

        # quality
        "all_fields_em": results["all_fields_em"].mean(),
        **{f"EM_{k}": results[f"em_{k}"].mean() for k in FIELDS},
        **{f"F1_{k}": results[f"f1_{k}"].mean() for k in FIELDS},

        # performance
        "avg_wall_s": results["wall_s"].mean(),
        "avg_prompt_tokens": results["prompt_tokens"].dropna().mean(),
        "avg_gen_tokens": results["gen_tokens"].dropna().mean(),
        "avg_prompt_tok_s": results["prompt_tok_s"].dropna().mean(),
        "avg_gen_tok_s": results["gen_tok_s"].dropna().mean(),
    }

    # behavior (separate)
    summary.update(behavior_metrics_receipts(results, fields=FIELDS))

    return results, summary


## 6. Task: Improving Receipt Information Extraction

Your goal is to improve the receipt information extraction system.

You may experiment with:
- different models
- different quantization levels
- prompt design
- generation parameters

You must keep:
- the dataset
- the evaluation code
- the metrics unchanged

### What to prioritize

In real-world receipt processing, the following matter most:

1. Correct extraction of critical fields (especially `total` and `date`)
2. Low hallucination (do not invent values)
3. Reasonable resource usage


### What to report

At the end of the notebook, report:

- the chosen model
- the final prompt
- the main metrics (EM, F1, runtime)
- a short justification explaining your trade-offs

## 7. Evaluating playground
Now you can evaluate and try your models, like in the example below.

In [ ]:
models_to_test = [
    "llama3.2:3b",
    #"llama3.2:3b-instruct-q8_0",
    #"llama3.1:8b-instruct-q4_K_M",
]

results_by_model = {}
summaries = []

for m in models_to_test:
    res, summ = run_receipt_task(m, items, n=50, stream=False, options={"temperature": 0.0, "num_predict": 200})
    results_by_model[m] = res
    summaries.append(summ)

pd.DataFrame(summaries)

In [ ]:
# Example of how to inspect errors
res[["gold_date", "pred_date", "gold_total", "pred_total", "gold_company", "pred_company"]]